# 05a — mechanism on the real networks

The interlayer-weight sweep (Figure 1), the variants (Table 3) with the classical grid, the partially observed test individuals (Table 4) with the label-free control, and the single-layer proposition. Every lumped operator goes through the router, and the route it took is recorded in the `route` column of each table that prints one.

The dataset is the parameters cell: `scripts/run_all.py` sets it for each dataset by replacing that cell in memory and keeps the executed copy as `outputs/executed_notebooks/<stage>_<dataset>.ipynb`; this tracked notebook stays stripped.


In [ ]:
# Parameter cell: scripts/run_all.py replaces it in memory for each dataset.
DATASET = "hmeq"


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path


def repo_root(start: Path | None = None) -> Path:
    """The qmulp checkout this notebook lives in."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "config.py").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("run this notebook from inside the qmulp checkout")


_root = repo_root()
sys.path.insert(0, str(_root / 'src'))
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import polars as pl
from IPython.display import display

from config import DEFAULT_CONFIG
from fingerprint import resolve_config, sha256_path
from paths import ROOT

pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(40)

# The frozen configuration this notebook runs with, echoed so the executed
# notebook records it. No absolute path is printed: artifacts are listed
# relative to the repository root.
config, smoke = resolve_config(DEFAULT_CONFIG)
print("dataset:", DATASET, "| smoke:", smoke)
print(json.dumps(config.as_dict(), indent=1, default=str))


## Run the stage


In [ ]:
from stages import mechanism

result = mechanism.run(DATASET, config)
display(pl.DataFrame(result["eta"], strict=False).to_pandas())
display(pl.DataFrame(result["variants"], strict=False).to_pandas())
display(pl.DataFrame(result["partial"], strict=False).to_pandas())
print(json.dumps(result["summary"]["single_layer"], indent=1, default=str))


## Artifacts


In [ ]:
for path in result["artifacts"]:
    print(f"{sha256_path(path)}  {Path(path).resolve().relative_to(ROOT)}")
